# NLP Movie Recommendation System — Level 2
### Weighted Similarity with Cast, Director, Genre, Keywords, and Plot

This notebook extends Level 1 by:

1. Adding **cast** (top 3 actors) and **director** from `tmdb_5000_credits.csv`
2. Building **separate TF-IDF vectors** for each feature: plot, genre, keywords, cast, director
3. Computing **cosine similarity per feature**, then combining them with a **weighted formula**:

```
Final Score =
    0.50 x Plot Similarity
  + 0.20 x Genre Similarity
  + 0.15 x Keyword Similarity
  + 0.10 x Cast Similarity
  + 0.05 x Director Similarity
```

4. Re-running the same Precision@5 evaluation to compare against the Level 1 baseline


In [1]:
import pandas as pd
import numpy as np
import ast
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

pd.set_option('display.max_colwidth', 100)


## 1. Load both datasets

`tmdb_5000_movies.csv` (plot/genre/keywords) and `tmdb_5000_credits.csv` (cast/crew), merged on movie id.

In [2]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

movies = movies[['id', 'title', 'genres', 'keywords', 'overview', 'vote_average', 'release_date']]
credits = credits.rename(columns={'movie_id': 'id'})[['id', 'cast', 'crew']]

movies = movies.merge(credits, on='id', how='left')
print("Shape after merge:", movies.shape)
movies.head(2)


Shape after merge: (4803, 9)


,id,title,genres,keywords,overview,vote_average,release_date,cast,crew
0,19995,Avatar,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""sp...","In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, ...",7.2,2009-12-10,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a3685532001c9a"", ""gender"": ...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender"": 0, ""id"": 1721, ""jo..."
1,285,Pirates of the Caribbean: At World's End,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic is...","Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of t...",6.9,2007-05-19,"[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c3a36847f800b50d"", ""g...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"": 2, ""id"": 120, ""job""..."


## 2. Clean missing values

In [3]:
movies = movies.dropna(subset=['overview']).reset_index(drop=True)
for col in ['keywords', 'genres', 'cast', 'crew']:
    movies[col] = movies[col].fillna('[]')

movies.isnull().sum()


id              0
title           0
genres          0
keywords        0
overview        0
vote_average    0
release_date    1
cast            0
crew            0
dtype: int64

## 3. Parse genres, keywords, cast, and director

- `genres` / `keywords` → list of names (same as Level 1)
- `cast` → names of the **top 3 billed actors** (by `order` field)
- `crew` → the **Director** (job == 'Director')

In [4]:
def parse_names(json_like_str, limit=None):
    try:
        items = ast.literal_eval(json_like_str)
        names = [d['name'] for d in items]
        return names[:limit] if limit else names
    except (ValueError, SyntaxError):
        return []

def parse_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        for member in crew:
            if member.get('job') == 'Director':
                return member['name']
    except (ValueError, SyntaxError):
        pass
    return ''

movies['genres_list'] = movies['genres'].apply(parse_names)
movies['keywords_list'] = movies['keywords'].apply(parse_names)
movies['cast_list'] = movies['cast'].apply(lambda x: parse_names(x, limit=3))
movies['director'] = movies['crew'].apply(parse_director)

movies[['title', 'genres_list', 'cast_list', 'director']].head(5)


,title,genres_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton


## 4. Text preprocessing for the plot overview

Same cleaning as Level 1: lowercase, strip punctuation, remove stopwords, lemmatize.

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

movies['overview_clean'] = movies['overview'].apply(clean_text)


## 5. Build one text field per feature

Unlike Level 1 (which merged everything into a single document), here we keep **five separate text fields**
so we can vectorize and score each one independently.

**Genre and keywords keep their natural spacing** (`"Science Fiction"` stays as two words, not squashed into
`"sciencefiction"`). This matters: a free-text query like *"science fiction space adventure"* is tokenized
by the same rules, so with `ngram_range=(1,2)` the bigram `"science fiction"` in the query can match the
bigram `"science fiction"` in a movie's genre field. Squashing into single tokens breaks that match entirely
since the query would need to say `"sciencefiction"` as one word to hit it.

**Cast and director names ARE squashed** (`"Christopher Nolan"` → `"christophernolan"`), because those are
matched separately via an exact-substring check against the query (see the recommend function below),
not through cosine similarity — so squashing there just avoids `"christopher"` accidentally matching an
unrelated actor's first name.

In [6]:
def join_spaced(names):
    return ' '.join(n.lower() for n in names)

def squash(names):
    return ' '.join(n.lower().replace(' ', '') for n in names)

movies['plot_text']     = movies['overview_clean']
movies['genre_text']    = movies['genres_list'].apply(join_spaced)
movies['keyword_text']  = movies['keywords_list'].apply(join_spaced)
movies['cast_text']     = movies['cast_list'].apply(squash)
movies['director_text'] = movies['director'].apply(lambda d: d.lower().replace(' ', ''))

movies[['plot_text', 'genre_text', 'keyword_text', 'cast_text', 'director_text']].head(3)


,plot_text,genre_text,keyword_text,cast_text,director_text
0,century paraplegic marine dispatched moon pandora unique mission becomes torn following order pr...,action adventure fantasy science fiction,culture clash future space war space colony society space travel futuristic romance space alien ...,samworthington zoesaldana sigourneyweaver,jamescameron
1,captain barbossa long believed dead come back life headed edge earth turner elizabeth swann noth...,adventure fantasy action,ocean drug abuse exotic island east india trading company love of one's life traitor shipwreck s...,johnnydepp orlandobloom keiraknightley,goreverbinski
2,cryptic message bond past sends trail uncover sinister organization battle political force keep ...,action adventure crime,spy based on novel secret agent sequel mi6 british secret service united kingdom,danielcraig christophwaltz léaseydoux,sammendes


## 6. Fit a separate TF-IDF vectorizer per feature

`ngram_range=(1,2)` on genre and keywords is what makes multi-word matches like "science fiction" work
as a phrase, while still allowing single-word matches like "space" or "comedy" to contribute too.

In [7]:
tfidf_plot     = TfidfVectorizer(max_features=15000, ngram_range=(1,2), min_df=2)
tfidf_genre    = TfidfVectorizer(ngram_range=(1,2))
tfidf_keyword  = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2)
tfidf_cast     = TfidfVectorizer(max_features=5000, min_df=1)
tfidf_director = TfidfVectorizer(max_features=3000, min_df=1)

matrix_plot     = tfidf_plot.fit_transform(movies['plot_text'])
matrix_genre    = tfidf_genre.fit_transform(movies['genre_text'])
matrix_keyword  = tfidf_keyword.fit_transform(movies['keyword_text'])
matrix_cast     = tfidf_cast.fit_transform(movies['cast_text'])
matrix_director = tfidf_director.fit_transform(movies['director_text'])

print("plot:", matrix_plot.shape)
print("genre:", matrix_genre.shape)
print("keyword:", matrix_keyword.shape)
print("cast:", matrix_cast.shape)
print("director:", matrix_director.shape)


plot: (4800, 15000)
genre: (4800, 276)
keyword: (4800, 5000)
cast: (4800, 5000)
director: (4800, 2465)


## 7. Weighted recommendation function

For a user query, we only vectorize it against the **plot**, **genre**, and **keyword** vectorizers
(a free-text query naturally maps to those three). Cast and director don't apply to a query directly —
they matter when the user names a specific actor/director in their query, so we also check for a
literal name match as a bonus signal.

Weights (from the project plan):

| Feature   | Weight |
|-----------|--------|
| Plot      | 50%    |
| Genre     | 20%    |
| Keywords  | 15%    |
| Cast      | 10%    |
| Director  | 5%     |


In [8]:
WEIGHTS = {
    'plot': 0.50,
    'genre': 0.20,
    'keyword': 0.15,
    'cast': 0.10,
    'director': 0.05,
}

def weighted_recommend(query, top_n=5, weights=WEIGHTS):
    query_clean = clean_text(query)
    query_squashed = query_clean.replace(' ', '')  # crude squash for name-style matching

    sim_plot = cosine_similarity(tfidf_plot.transform([query_clean]), matrix_plot).flatten()
    sim_genre = cosine_similarity(tfidf_genre.transform([query_clean]), matrix_genre).flatten()
    sim_keyword = cosine_similarity(tfidf_keyword.transform([query_clean]), matrix_keyword).flatten()

    # Cast / director: reward an exact substring name match in the query (e.g. "Christopher Nolan movie")
    query_lower = query.lower().replace(' ', '')
    sim_cast = movies['cast_text'].apply(
        lambda names: 1.0 if names and any(n in query_lower for n in names.split()) else 0.0
    ).to_numpy()
    sim_director = movies['director_text'].apply(
        lambda d: 1.0 if d and d in query_lower else 0.0
    ).to_numpy()

    final_score = (
        weights['plot'] * sim_plot +
        weights['genre'] * sim_genre +
        weights['keyword'] * sim_keyword +
        weights['cast'] * sim_cast +
        weights['director'] * sim_director
    )

    result = movies.copy()
    result['score'] = final_score
    result = result.sort_values('score', ascending=False).head(top_n)
    result['match_%'] = (result['score'] * 100).round(1)
    return result[['title', 'genres_list', 'director', 'cast_list', 'vote_average', 'match_%']].reset_index(drop=True)


## 8. Try it

In [9]:
weighted_recommend("science fiction space adventure with an emotional story", top_n=5)


,title,genres_list,director,cast_list,vote_average,match_%
0,Gattaca,"[Thriller, Science Fiction, Mystery, Romance]",Andrew Niccol,"[Ethan Hawke, Jude Law, Gore Vidal]",7.5,27.2
1,Ender's Game,"[Science Fiction, Action, Adventure]",Gavin Hood,"[Asa Butterfield, Harrison Ford, Hailee Steinfeld]",6.6,23.4
2,"The Beast from 20,000 Fathoms","[Adventure, Horror, Science Fiction]",Eugène Lourié,"[Paul Hubschmid, Paula Raymond, Cecil Kellaway]",6.7,22.0
3,Flatliners,"[Drama, Horror, Science Fiction, Thriller]",Joel Schumacher,"[Kiefer Sutherland, Julia Roberts, Kevin Bacon]",6.3,21.8
4,Contact,"[Drama, Science Fiction, Mystery]",Robert Zemeckis,"[Jodie Foster, Matthew McConaughey, James Woods]",7.2,21.5


In [10]:
weighted_recommend("Christopher Nolan science fiction movie about space", top_n=5)


,title,genres_list,director,cast_list,vote_average,match_%
0,Gattaca,"[Thriller, Science Fiction, Mystery, Romance]",Andrew Niccol,"[Ethan Hawke, Jude Law, Gore Vidal]",7.5,22.9
1,Flatliners,"[Drama, Horror, Science Fiction, Thriller]",Joel Schumacher,"[Kiefer Sutherland, Julia Roberts, Kevin Bacon]",6.3,17.7
2,Contact,"[Drama, Science Fiction, Mystery]",Robert Zemeckis,"[Jodie Foster, Matthew McConaughey, James Woods]",7.2,17.4
3,Mars Attacks!,"[Comedy, Fantasy, Science Fiction]",Tim Burton,"[Jack Nicholson, Glenn Close, Annette Bening]",6.1,17.0
4,The Last Days on Mars,"[Science Fiction, Thriller, Horror]",Ruairi Robinson,"[Liev Schreiber, Romola Garai, Elias Koteas]",5.2,16.9


In [11]:
weighted_recommend("funny movie about friendship and college life", top_n=5)


,title,genres_list,director,cast_list,vote_average,match_%
0,Ask Me Anything,"[Drama, Mystery, Thriller]",Allison Burnett,"[Britt Robertson, Christian Slater, Justin Long]",5.5,13.4
1,Disaster Movie,"[Action, Comedy]",Jason Friedberg,"[Matt Lanter, Vanessa Lachey, Nicole Ari Parker]",3.0,11.8
2,Sharknado,"[TV Movie, Horror]",Anthony C. Ferrante,"[Ian Ziering, Tara Reid, Cassandra Scerbo]",3.8,10.7
3,October Baby,[Drama],Andrew Erwin,"[Rachel Hendrix , Jason Burkey, Robert Amaya ]",6.8,10.0
4,We Have Your Husband,"[TV Movie, Crime, Drama, Thriller]",Eric Bross,"[Teri Polo, Esai Morales, Nicholas Gonzalez]",5.0,10.0


## 9. Compare Level 1 vs Level 2 — Precision@5

Same 7 test queries as Level 1, now scored with the weighted multi-feature engine.

In [12]:
test_queries = [
    ("science fiction space adventure", "Science Fiction"),
    ("romantic comedy", "Romance"),
    ("superhero action movie", "Action"),
    ("horror movie with ghosts", "Horror"),
    ("animated family movie", "Animation"),
    ("crime thriller", "Crime"),
    ("historical drama", "Drama"),
]

def precision_at_k(query, relevant_genre, k=5):
    results = weighted_recommend(query, top_n=k)
    hits = results['genres_list'].apply(lambda genres: relevant_genre in genres)
    return hits.sum() / k

scores = []
for query, genre in test_queries:
    p = precision_at_k(query, genre, k=5)
    scores.append(p)
    print(f"Query: {query!r:45s} | Target genre: {genre:16s} | Precision@5 = {p:.2f}")

print(f"\nAverage Precision@5 (Level 2, weighted): {np.mean(scores):.2f}")
print("Average Precision@5 (Level 1, plain TF-IDF): 0.71   <- from the Level 1 notebook")


Query: 'science fiction space adventure'             | Target genre: Science Fiction  | Precision@5 = 1.00
Query: 'romantic comedy'                             | Target genre: Romance          | Precision@5 = 0.20
Query: 'superhero action movie'                      | Target genre: Action           | Precision@5 = 0.80
Query: 'horror movie with ghosts'                    | Target genre: Horror           | Precision@5 = 1.00
Query: 'animated family movie'                       | Target genre: Animation        | Precision@5 = 0.40
Query: 'crime thriller'                              | Target genre: Crime            | Precision@5 = 1.00
Query: 'historical drama'                            | Target genre: Drama            | Precision@5 = 1.00

Average Precision@5 (Level 2, weighted): 0.77
Average Precision@5 (Level 1, plain TF-IDF): 0.71   <- from the Level 1 notebook


## 10. Experiment: does the weighting actually help?

Try a couple of alternate weight sets and see how Precision@5 changes.
This is good material for your report — you can show *why* you picked the final weights.

In [13]:
weight_variants = {
    "Plan default (50/20/15/10/5)": {'plot':0.50,'genre':0.20,'keyword':0.15,'cast':0.10,'director':0.05},
    "Plot-heavy (70/10/10/5/5)":     {'plot':0.70,'genre':0.10,'keyword':0.10,'cast':0.05,'director':0.05},
    "Genre-heavy (30/40/15/10/5)":   {'plot':0.30,'genre':0.40,'keyword':0.15,'cast':0.10,'director':0.05},
}

for name, w in weight_variants.items():
    scores = []
    for query, genre in test_queries:
        results = weighted_recommend(query, top_n=5, weights=w)
        hits = results['genres_list'].apply(lambda genres: genre in genres)
        scores.append(hits.sum() / 5)
    print(f"{name:32s} -> Avg Precision@5 = {np.mean(scores):.2f}")


Plan default (50/20/15/10/5)     -> Avg Precision@5 = 0.77
Plot-heavy (70/10/10/5/5)        -> Avg Precision@5 = 0.74
Genre-heavy (30/40/15/10/5)      -> Avg Precision@5 = 0.66


## 11. Save Level 2 artifacts

Separate from the Level 1 artifacts, so the Streamlit app can offer both modes.

In [14]:
import pickle

artifacts_v2 = {
    'movies': movies,
    'tfidf_plot': tfidf_plot, 'matrix_plot': matrix_plot,
    'tfidf_genre': tfidf_genre, 'matrix_genre': matrix_genre,
    'tfidf_keyword': tfidf_keyword, 'matrix_keyword': matrix_keyword,
    'tfidf_cast': tfidf_cast, 'matrix_cast': matrix_cast,
    'tfidf_director': tfidf_director, 'matrix_director': matrix_director,
    'weights': WEIGHTS,
}

with open('movies_v2.pkl', 'wb') as f:
    pickle.dump(artifacts_v2, f)

print("Saved: movies_v2.pkl (contains all Level 2 artifacts)")


Saved: movies_v2.pkl (contains all Level 2 artifacts)


## Summary

- Level 1 used a single merged TF-IDF over genre+keywords+overview.
- Level 2 scores plot, genre, keywords, cast, and director **independently**, then blends them with
  explicit weights — which is more explainable ("why recommended" can now say *how much* each factor
  contributed) and lets you tune the weights based on measured Precision@5.
- Next: wire `weighted_recommend()` and `movies_v2.pkl` into the Streamlit app as a second mode
  ("Basic TF-IDF" vs "Weighted (Level 2)"), with a slider UI for the weights.
